## 1. SparkSession - entry point

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, StringType)

import pandas as pd
import matplotlib.pyplot as plt

import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
        .appName("NinjaTraderETL")
        .master("local[*]")  # use all cores on machine as workers
        .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11")
        .getOrCreate()
)

spark

## 2. Extract — read the raw CSV with an explicit schema (bronze layer)

In [ ]:
raw_schema = StructType([
    StructField("trade_number", IntegerType(), True),
    StructField("qty", IntegerType(), True),
    StructField("entry_price", DoubleType(), True),
    StructField("exit_price", DoubleType(), True),
    StructField("entry_time_raw", StringType(), True),
    StructField("exit_time_raw", StringType(), True),
    StructField("entry_name", StringType(), True),
    StructField("exit_name", StringType(), True),
    StructField("profit_raw", StringType(), True),
    StructField("cum_net_profit_raw", StringType(), True),
    StructField("mae_raw", StringType(), True),
    StructField("mfe_raw", StringType(), True),
    StructField("bars", IntegerType(), True),
    StructField("_trailing", StringType(), True) # ninjatrader exports a trailing comma
])

bronze_df = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv("trades.csv")
)

print(f"row count: {bronze_df.count()}")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

## 3. Transform — clean types, parse currency, derive columns (silver layer)

In [ ]:
def parse_currency(colname: str):
    """convert ninjatrader-formatted currency stirngs like '$850.00' / '($850.00)' to signed double"""
    c = F.col(colname)
    is_negative = c.startswith("(")
    stripped = F.regexp_replace(c, r"[\$,()]","")
    numeric = stripped.cast("double")
    return F.when(is_negative, -numeric).otherwise(numeric)

TS_FORMAT = "M/d/yyyy h:mm:ss a"

silver_df = (
    bronze_df
    .withColumn("entry_time", F.to_timestamp("entry_time_raw", TS_FORMAT))
    .withColumn("exit_time", F.to_timestamp("exit_time_raw", TS_FORMAT))
    .withColumn("profit", parse_currency("profit_raw"))
    .withColumn("mae", parse_currency("mae_raw"))
    .withColumn("mfe", parse_currency("mfe_raw"))
    .withColumn(
        "side",
        F.when(F.col("entry_name").contains("long"), "long")
         .when(F.col("entry_name").contains("short"), "short")
         .otherwise("unknown")
    )
    .withColumn(
        "duration_minutes",
        (F.col("exit_time").cast("long") - F.col("entry_time").cast("long")) / 60.0
    )
    .withColumn("is_win", F.col("profit") > 0)
    .select(
        "trade_number", "qty", "side", "entry_price", "exit_price", "entry_time", "exit_time", "duration_minutes",
        "bars", "exit_name", "profit", "mae", "mfe", "is_win"
    )
)

silver_df.printSchema()
silver_df.orderBy("trade_number").show(10, truncate=False)

## 4. Gold layer — window functions for cumulative P&L and drawdown

In [ ]:
trade_order = Window.orderBy("trade_number").rowsBetween(Window.unboundedPreceding, Window.currentRow)

gold_df = (
    silver_df
    .withColumn("cum_net_profit", F.round(F.sum("profit").over(trade_order), 2))
    .withColumn("running_max_equity", F.max("cum_net_profit").over(trade_order))
    .withColumn("drawdown",  F.round(F.col("cum_net_profit") - F.col("running_max_equity"), 2))
)

gold_df.select(
    "trade_number", "entry_time", "side", "profit", "cum_net_profit", "drawdown"
).orderBy("trade_number").show(10, truncate=False)

max_drawdown = gold_df.agg(F.min("drawdown")).first()[0]
print(f"max drawdown (spark-computed): ${max_drawdown:,.2f}")

## 5. Summary statistics — groupBy / agg

In [ ]:
summary_by_side = (
    gold_df.groupBy("side")
    .agg(
        F.count("*").alias("num_trades"),
        F.round(F.avg(F.col("is_win").cast("double")) * 100, 2).alias("win_rate_pct"),
        F.round(F.sum(F.when(F.col("profit") > 0, F.col("profit")).otherwise(0.0)), 2).alias("gross_profit"),
        F.round(F.sum(F.when(F.col("profit") < 0, F.col("profit")).otherwise(0.0)), 2).alias("gross_loss"),
        F.round(F.sum("profit"), 2).alias("net_profit"),
        F.round(F.avg(F.when(F.col("profit") > 0, F.col("profit"))), 2).alias("avg_win"),
        F.round(F.avg(F.when(F.col("profit") < 0, F.col("profit"))), 2).alias("avg_loss"),
    )
    .withColumn("profit_factor", F.round(F.col("gross_profit") / F.abs(F.col("gross_loss")), 2))
    .orderBy("side")
)

summary_by_side.show(truncate=False)

overall = gold_df.agg(
    F.count("*").alias("num_trades"),
    F.round(F.avg(F.col("is_win").cast("double")) * 100, 2).alias("win_rate_pct"),
    F.round(F.sum("profit"), 2).alias("net_profit")
)

overall.show(truncate=False)

gold_df.createOrReplaceTempView("trades")

spark.sql("""
    select side, count(*) as num_trades, round(avg(case when is_win then 1.0 else 0.0 end) * 100, 2) as win_rate_pct,
        round(sum(profit),2) as net_profit
    from trades
    group by side
    order by side
""").show(truncate=False)

## 6. Load — write the Gold table out

In [ ]:
(
    gold_df
    .write
    .mode("overwrite")
    .partitionBy("side")
    .parquet("output/trades_gold")
)

print("wrote gold layer to output/trades_gold/ (partitioned by side (long or short)")

spark.read.parquet("output/trades_gold").count()

# write to local sql server docker

from dotenv import load_dotenv
import os

load_dotenv()  # reads .env in the current working directory into environment variables

sqlserver_host = os.getenv("SQLSERVER_HOST")
sqlserver_port = os.getenv("SQLSERVER_PORT")
sqlserver_db = os.getenv("SQLSERVER_DB")
sqlserver_user = os.getenv("SQLSERVER_USER")
sqlserver_password = os.getenv("SQLSERVER_PASSWORD")

pw = os.getenv("SQLSERVER_PASSWORD")
print(len(pw), pw[0], pw[-1])  # sanity-check password retrieval

jdbc_url = (
    f"jdbc:sqlserver://{sqlserver_host}:{sqlserver_port};"
    f"databaseName={sqlserver_db};"
    "trustServerCertificate=true"  # needed for local Docker SQL Server's self-signed cert
)

(
    gold_df
    .write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.trades_gold")
    .option("user", sqlserver_user)
    .option("password", sqlserver_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("overwrite")
    .save()
)

print("Wrote gold_df to dbo.trades_gold via JDBC")

## 7. Reproduce the NinjaTrader charts

In [ ]:
plot_pdf = (
    gold_df
    .select("trade_number", "entry_time", "cum_net_profit", "drawdown")
    .orderBy("trade_number")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(plot_pdf["entry_time"], plot_pdf["cum_net_profit"], color="#2ca02c", alpha=0.6)
ax.plot(plot_pdf["entry_time"], plot_pdf["cum_net_profit"], color="#2ca02c", linewidth=1)
ax.set_title("Cumulative Net Profit (Spark-computed)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative profit ($)")
ax.axhline(0, color="white", linewidth=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(plot_pdf["entry_time"], plot_pdf["drawdown"], color="#d62728", alpha=0.6)
ax.plot(plot_pdf["entry_time"], plot_pdf["drawdown"], color="#d62728", linewidth=1)
ax.set_title("Cumulative Max Drawdown (Spark-computed)")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown ($)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Max drawdown: ${plot_pdf['drawdown'].min():,.2f}   (NinjaTrader reported: ($2,750.00))")